# Лабораторная работа 9

Численное решение линейного уравнения параболического типа

Вариант 21:

- `U_t = D U_xx`
- `D = 1`
- `U(0,x)=2-x`
- `U(t,0)=2`
- `U(t,1)=1`
- область: `x ∈ [0,1]`, `t ∈ [0,10]`


In [1]:
import contextlib
import io
import json
from pathlib import Path

import matplotlib
import numpy as np
from docx import Document
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.shared import Inches, Pt
from matplotlib import pyplot as plt

matplotlib.use("Agg")

BASE_DIR = Path(__file__).resolve().parent
NOTEBOOK_PATH = BASE_DIR / "9.ipynb"
DOCX_PATH = BASE_DIR / "9_Туманов.docx"
PLOTS_DIR = BASE_DIR / "plots"

X0 = 0.0
X1 = 1.0
T0 = 0.0
T1 = 10.0
D = 1.0
EPS = 0.01
H = 0.1
TAU = 0.005
N = int(round((X1 - X0) / H))
M = int(round((T1 - T0) / TAU))
R = D * TAU / (H * H)

X = np.linspace(X0, X1, N + 1)
T = np.linspace(T0, T1, M + 1)


def phi(x):
    return 2.0 - x


def left_boundary(t):
    return 2.0


def right_boundary(t):
    return 1.0


def exact_solution(x, t):
    return 2.0 - x


def init_grid():
    u = np.zeros((M + 1, N + 1))
    u[0, :] = phi(X)
    u[:, 0] = left_boundary(T)
    u[:, N] = right_boundary(T)
    return u


def solve_explicit():
    u = init_grid()
    for n in range(M):
        for j in range(1, N):
            u[n + 1, j] = u[n, j] + R * (u[n, j - 1] - 2.0 * u[n, j] + u[n, j + 1])
        u[n + 1, 0] = left_boundary(T[n + 1])
        u[n + 1, N] = right_boundary(T[n + 1])
    return u


def thomas(a, b, c, f):
    size = len(f)
    alpha = np.zeros(size)
    beta = np.zeros(size)
    y = np.zeros(size)

    alpha[0] = -c[0] / b[0]
    beta[0] = f[0] / b[0]
    for i in range(1, size):
        denominator = b[i] + a[i] * alpha[i - 1]
        alpha[i] = 0.0 if i == size - 1 else -c[i] / denominator
        beta[i] = (f[i] - a[i] * beta[i - 1]) / denominator

    y[-1] = beta[-1]
    for i in range(size - 2, -1, -1):
        y[i] = alpha[i] * y[i + 1] + beta[i]
    return y


def solve_implicit():
    u = init_grid()
    size = N - 1
    for n in range(M):
        a = np.full(size, -R)
        b = np.full(size, 1.0 + 2.0 * R)
        c = np.full(size, -R)
        f = np.array([u[n, j] for j in range(1, N)])

        f[0] -= a[0] * left_boundary(T[n + 1])
        a[0] = 0.0
        f[-1] -= c[-1] * right_boundary(T[n + 1])
        c[-1] = 0.0

        inner = thomas(a, b, c, f)
        u[n + 1, 0] = left_boundary(T[n + 1])
        u[n + 1, N] = right_boundary(T[n + 1])
        u[n + 1, 1:N] = inner
    return u


def exact_grid():
    u = np.zeros((M + 1, N + 1))
    for n, tt in enumerate(T):
        u[n, :] = exact_solution(X, tt)
    return u


def max_abs_error(u, exact):
    return float(np.max(np.abs(u - exact)))


def sample_row(exact, explicit, implicit, time_value=0.1):
    index = int(round(time_value / TAU))
    rows = []
    for j, x in enumerate(X):
        rows.append(
            {
                "x": x,
                "exact": exact[index, j],
                "explicit": explicit[index, j],
                "implicit": implicit[index, j],
            }
        )
    return rows


def plot_heatmap(u, title, filename):
    PLOTS_DIR.mkdir(exist_ok=True)
    x_mesh, t_mesh = np.meshgrid(X, T)
    fig, ax = plt.subplots(figsize=(8, 5.5))
    im = ax.contourf(x_mesh, t_mesh, u, levels=50, cmap="viridis")
    ax.set_xlabel("x")
    ax.set_ylabel("t")
    ax.set_title(title)
    fig.colorbar(im, ax=ax, label="U(x,t)")
    fig.tight_layout()
    path = PLOTS_DIR / filename
    fig.savefig(path, dpi=150)
    plt.close(fig)
    return path


def plot_surface(u, title, filename):
    PLOTS_DIR.mkdir(exist_ok=True)
    x_mesh, t_mesh = np.meshgrid(X, T)
    step_t = max(1, len(T) // 120)
    fig = plt.figure(figsize=(8.5, 6))
    ax = fig.add_subplot(111, projection="3d")
    surf = ax.plot_surface(
        x_mesh[::step_t],
        t_mesh[::step_t],
        u[::step_t],
        cmap="viridis",
        edgecolor="none",
    )
    ax.set_xlabel("x")
    ax.set_ylabel("t")
    ax.set_zlabel("U(x,t)")
    ax.set_title(title)
    ax.view_init(elev=25, azim=-45)
    fig.colorbar(surf, shrink=0.55, aspect=12)
    fig.tight_layout()
    path = PLOTS_DIR / filename
    fig.savefig(path, dpi=150)
    plt.close(fig)
    return path


def plot_profile(exact, explicit, implicit, time_value, filename):
    PLOTS_DIR.mkdir(exist_ok=True)
    index = int(round(time_value / TAU))
    fig, ax = plt.subplots(figsize=(8.5, 5.5))
    ax.plot(X, exact[index], "k-", linewidth=2.5, label="Точное решение")
    ax.plot(X, explicit[index], "b--o", linewidth=2, markersize=5, label="Явная схема")
    ax.plot(X, implicit[index], "r-.s", linewidth=2, markersize=5, label="Неявная схема")
    ax.set_xlabel("x")
    ax.set_ylabel(f"U(x,{time_value})")
    ax.set_title(f"Сравнение решений при t = {time_value}")
    ax.grid(True, alpha=0.35)
    ax.legend()
    fig.tight_layout()
    path = PLOTS_DIR / filename
    fig.savefig(path, dpi=150)
    plt.close(fig)
    return path


def compute_results():
    exact = exact_grid()
    explicit = solve_explicit()
    implicit = solve_implicit()
    errors = {
        "Явная схема": max_abs_error(explicit, exact),
        "Неявная схема": max_abs_error(implicit, exact),
    }
    samples = sample_row(exact, explicit, implicit, 0.1)
    return exact, explicit, implicit, errors, samples


def create_plots(exact, explicit, implicit):
    return [
        plot_heatmap(exact, "Точное решение", "heat_exact.png"),
        plot_heatmap(explicit, "Явная схема", "heat_explicit.png"),
        plot_heatmap(implicit, "Неявная схема", "heat_implicit.png"),
        plot_profile(exact, explicit, implicit, 0.1, "profile_t01.png"),
        plot_surface(explicit, "Явная схема", "surface_explicit.png"),
        plot_surface(implicit, "Неявная схема", "surface_implicit.png"),
    ]



In [2]:
exact, explicit, implicit, errors, samples = compute_results()
plots = create_plots(exact, explicit, implicit)

print(f'Сетка: h={H}, tau={TAU}, N={N}, M={M}, r={R:.3f}')
print('Максимальные абсолютные погрешности:')
for name, value in errors.items():
    print(f'{name}: {value:.6f}')
print()
print('Сравнение решений при t=0.1:')
print(f"{'x':>6} {'exact':>10} {'explicit':>10} {'implicit':>10}")
for row in samples:
    print(f"{row['x']:6.1f} {row['exact']:10.3f} {row['explicit']:10.3f} {row['implicit']:10.3f}")
print()
print('Графики:')
for plot in plots:
    print(plot)


Сетка: h=0.1, tau=0.005, N=10, M=2000, r=0.500
Максимальные абсолютные погрешности:
Явная схема: 0.000000
Неявная схема: 0.000000

Сравнение решений при t=0.1:
     x      exact   explicit   implicit
   0.0      2.000      2.000      2.000
   0.1      1.900      1.900      1.900
   0.2      1.800      1.800      1.800
   0.3      1.700      1.700      1.700
   0.4      1.600      1.600      1.600
   0.5      1.500      1.500      1.500
   0.6      1.400      1.400      1.400
   0.7      1.300      1.300      1.300
   0.8      1.200      1.200      1.200
   0.9      1.100      1.100      1.100
   1.0      1.000      1.000      1.000

Графики:
C:\GitHub\BachelorLabs\NumericalMethods\Semester-6\Lab9\plots\heat_exact.png
C:\GitHub\BachelorLabs\NumericalMethods\Semester-6\Lab9\plots\heat_explicit.png
C:\GitHub\BachelorLabs\NumericalMethods\Semester-6\Lab9\plots\heat_implicit.png
C:\GitHub\BachelorLabs\NumericalMethods\Semester-6\Lab9\plots\profile_t01.png
C:\GitHub\BachelorLabs\NumericalMet